# A3.3 · Egress control

**Function A — AI Architecture, Risks and Mitigations → Controls — Runtime and the Gateway**  ·  *Security of AI*

Builds on **[A3.2 · Sandboxed execution](https://spbreed.github.io/cyber-commons/lessons/A3.2.html)**.

| | |
|---|---|
| Open-source tooling | Cilium, agentgateway |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The hook

Every exfiltration path in the architecture ends at the same place: a packet leaving your network. That makes egress the highest-leverage control you have, and the one most often left as allow-all because it broke something once.

## 2 · The framework

```
   every exfiltration path, whatever its start, ends here:

   prompt leak  --+
   tool abuse   --+---> data assembled ---> [ egress ] ---> out
   code exec    --+                            ^
   memory read  --+                            |
                                    one place to enforce, one to log

   allow-all egress makes every control upstream best-effort
```

**Mitigates: T2 Tool Misuse · T6 Intent Breaking · LLM02 Sensitive Information Disclosure.**

Egress is the highest-leverage control in the architecture, for a structural
reason: **every exfiltration path ends at the network boundary**, no matter how
the agent was persuaded to take it.

Injection, tool misuse, a compromised MCP server, model-authored code, a
poisoned peer message — all of them converge on the same final step. Data leaves.
A control at that step does not need to understand what happened upstream, which
is exactly what makes it robust: it is the one place where you do not have to
predict the attack.

Two rules that decide whether it works:

**Allow-list, never deny-list.** You cannot enumerate the internet. A deny-list
blocks the destinations you thought of.

**Specific destinations.** `*.googleapis.com` or `*.s3.amazonaws.com` is not an
egress policy — anyone can create a bucket in those namespaces, and A1.3's
attacker will. The allow-list holds the destinations this workload actually
needs, and there are usually fewer than five.

And one placement rule: enforce it **where the agent cannot rewrite it** — the
network layer, the sidecar, the gateway. An egress check inside the agent's own
process is a suggestion to the component being attacked.

The cost is honest: an agent that needs the open internet cannot have this
control, and that is a design decision to make deliberately rather than by
default.

> **What this control closes.**
>
> The one control that does not need to know how the attack worked, because every exfiltration path ends here.

## 2 · The control

In [ ]:
ALLOW = {"api.corp.example", "reports-db.corp.example"}
DENY_SUFFIXES = {".evil.example"}          # the deny-list, for comparison

def by_denylist(host):
    return not any(host.endswith(s) for s in DENY_SUFFIXES)

def by_allowlist(host):
    return host in ALLOW                    # exact, not suffix

DESTINATIONS = [
 ("api.corp.example",              "the one it actually needs"),
 ("archive.evil.example",          "A1.3's exfiltration target"),
 ("attacker-bucket.s3.amazonaws.com", "a bucket anyone can create"),
 ("169.254.169.254",               "cloud metadata - every credential"),
 ("pastebin.example",              "not on anyone's deny-list"),
]

print(f"{'destination':38s}{'deny-list':12s}{'allow-list':12s}note")
for host, note in DESTINATIONS:
    d, a = by_denylist(host), by_allowlist(host)
    print(f"{host:38s}{'allow' if d else 'block':12s}{'allow' if a else 'block':12s}{note}")

leaked = [h for h, _ in DESTINATIONS if by_denylist(h) and h not in ALLOW]
print(f"\ndeny-list lets through : {len(leaked)}  {leaked}")
print(f"allow-list lets through : {sorted(h for h, _ in DESTINATIONS if by_allowlist(h))}")
print()
print("The deny-list blocked exactly the destination somebody had already")
print("thought of. It cannot be completed, because the internet cannot be")
print("enumerated.")
print()
print("Placement matters as much: this check belongs in the network path, not")
print("in the agent. A check inside the process being attacked is advice.")
assert len(leaked) == 3 and len([h for h, _ in DESTINATIONS if by_allowlist(h)]) == 1

## What you just proved

Five destinations are evaluated both ways. The deny-list permits three exfiltration paths — a public-cloud bucket namespace anyone can register in, the cloud metadata address, and a host nobody listed — while the exact allow-list permits only the one destination the workload needs.

## Your turn

Write the allow-list for one agent by listing the hosts it genuinely calls. If it is under five, you can ship this control this week; if it is unbounded, that is the finding.

---

**Next → [A3.4 · Budgets and stop conditions](https://spbreed.github.io/cyber-commons/lessons/A3.4.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A3.3.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A3.3.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*